In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Reshape
from skimage.feature import hog
import tensorflow as tf
import keras
from tensorflow.keras import layers

In [ ]:
train_dir = '/content/drive/MyDrive/datasplit/Train'
val_dir = '/content/drive/MyDrive/datasplit/Test'

batch_size = 32
img_size = (64,64)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 4355 images belonging to 36 classes.
Found 1801 images belonging to 36 classes.


In [ ]:

def extract_hog_features(img):
    fd, hog_image = hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True, multichannel=True)
    return fd

def get_data_generator(generator):
    while True:
        x_batch, y_batch = generator.next()
        x_batch_hog = np.zeros((x_batch.shape[0], 1764))  # 64x64x3 -> 7x7x36
        for i in range(x_batch.shape[0]):
            x_batch_hog[i] = extract_hog_features(x_batch[i])
        yield x_batch_hog, y_batch

train_hog_generator = get_data_generator(train_generator)
val_hog_generator = get_data_generator(val_generator)


###Sample2 copy

In [ ]:
#sample1

model = Sequential()
#model.add(Reshape((42, 42, 1), input_shape=(1764,)))
model.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(64, 64, 3)))
model.add(Conv2D(32, (3, 3), padding='same', activation='relu'))
model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.1))

model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(MaxPooling2D((2, 2)))
#model.add(Dropout(0.1))

model.add(Flatten())
model.add(Dense(218, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(36, activation='softmax'))


In [ ]:

# compile the model with appropriate optimizer, loss function, and metrics
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:

# define callbacks to prevent overfitting
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=15, verbose=1, mode='min')
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1, mode='min', min_lr=0.0001)


In [ ]:

# fit the model with the training dataset and validate it with the testing dataset
history = model.fit(train_generator, epochs=15, validation_data=val_generator, callbacks=[early_stop, reduce_lr])

Epoch 1/15
137/137 [==============================] - 154s 1s/step - loss: 2.4897 - accuracy: 0.3160 - val_loss: 2.0432 - val_accuracy: 0.5875 - lr: 0.0010
Epoch 2/15
137/137 [==============================] - 147s 1s/step - loss: 0.3085 - accuracy: 0.9162 - val_loss: 2.1344 - val_accuracy: 0.6074 - lr: 0.0010
Epoch 3/15
137/137 [==============================] - 147s 1s/step - loss: 0.1288 - accuracy: 0.9621 - val_loss: 2.0726 - val_accuracy: 0.5586 - lr: 0.0010
Epoch 4/15
137/137 [==============================] - 143s 1s/step - loss: 0.0859 - accuracy: 0.9741 - val_loss: 1.9669 - val_accuracy: 0.6519 - lr: 0.0010
Epoch 5/15
137/137 [==============================] - 145s 1s/step - loss: 0.0560 - accuracy: 0.9816 - val_loss: 1.9048 - val_accuracy: 0.6830 - lr: 0.0010
Epoch 6/15
137/137 [==============================] - 144s 1s/step - loss: 0.0391 - accuracy: 0.9878 - val_loss: 2.3059 - val_accuracy: 0.6385 - lr: 0.0010
Epoch 7/15
137/137 [==============================] - 147s 1s/st

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.savefig("resenet50_with_augmentation.png")
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.savefig("resenet50_with_augmentation_loss.png")
plt.show()